In [12]:
import pandas as pd

# the handover file from notebook 2: 81 features + target, no dates
df = pd.read_parquet('../data/processed/model_table.parquet')

# the same source notebook 2 started from, so we can recover the dates
raw = pd.read_parquet('../data/processed/sample_50k.parquet')

# repeat notebook 2's row filter exactly, so the rows line up
good = ['Fully Paid']
bad  = ['Charged Off', 'Default']
keep = raw['loan_status'].isin(good + bad)

labelled = raw[keep].copy().reset_index(drop=True)

print(f"model_table rows: {len(df)}")
print(f"refiltered rows : {len(labelled)}")

# position-matching is only valid if both sides are the same length
assert len(labelled) == len(df)

df['issue_d'] = labelled['issue_d']
df['earliest_cr_line'] = labelled['earliest_cr_line']

print(df[['issue_d', 'earliest_cr_line']].head())
print(df.shape)



model_table rows: 29619
refiltered rows : 29619
    issue_d earliest_cr_line
0  Dec-2015         Dec-2003
1  Dec-2015         Oct-1976
2  Dec-2015         Nov-2009
3  Dec-2015         Mar-2006
4  Dec-2015         Dec-1982
(29619, 84)


In [17]:
df['issue_d'].head(5)
# df['issue_d'].isna().sum()
df['issue_d'].isna().mean()

np.float64(0.0)

In [24]:
# turn the text date into a real date ("Dec-2015" -> 2015-12-01)
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')

# the date with 80% of loans before it and the newest 20% after
cut = df['issue_d'].quantile(0.8)
print('cut-off date:', cut.date())

train = df[df['issue_d'] <  cut].copy()
test  = df[df['issue_d'] >= cut].copy()

print(f"train rows: {len(train):6d}   share: {len(train)/len(df):.1%}   bad rate: {train['target'].mean():.3%}")
print(f"test  rows: {len(test):6d}   share: {len(test)/len(df):.1%}   bad rate: {test['target'].mean():.3%}")

# nothing lost, nothing duplicated
assert len(train) + len(test) == len(df)
print('date range train:', train['issue_d'].min().date(), '->', train['issue_d'].max().date())
print('date range test :', test['issue_d'].min().date(), '->', test['issue_d'].max().date())

cut-off date: 2016-10-01
train rows:  23399   share: 79.0%   bad rate: 19.586%
test  rows:   6220   share: 21.0%   bad rate: 21.720%
date range train: 2007-07-01 -> 2016-09-01
date range test : 2016-10-01 -> 2018-12-01


In [25]:
by_year = df.groupby(df['issue_d'].dt.year).agg(
    rows        = ('target', 'size'),
    bad_rate    = ('target', 'mean'),
    share_60m   = ('term',   lambda s: (s == 60).mean()),
)
by_year['bad_rate']  = (by_year['bad_rate']  * 100).round(2)
by_year['share_60m'] = (by_year['share_60m'] * 100).round(1)
print(by_year)

# the same, split by term, for the test years only
print(
    df[df['issue_d'] >= '2016-10-01']
      .groupby([df['issue_d'].dt.year, 'term'])['target']
      .agg(['size', 'mean'])
)

         rows  bad_rate  share_60m
issue_d                           
2007        9     33.33        0.0
2008       32     15.62        0.0
2009      112     13.39        0.0
2010      269     10.04       31.2
2011      503     14.51       34.4
2012     1180     14.66       18.3
2013     2955     15.13       26.0
2014     4951     18.76       28.2
2015     8228     20.35       25.7
2016     6438     24.23       20.8
2017     3708     22.98       23.2
2018     1234     14.26       26.7
              size      mean
issue_d term                
2016    36.0  1004  0.225100
        60.0   274  0.354015
2017    36.0  2847  0.201616
        60.0   861  0.322880
2018    36.0   905  0.118232
        60.0   329  0.209726


In [27]:
# --- decision: restrict to 2013-2017 -------------------------------------
# 2018 : only 1,234 finished loans, and they are mostly early prepayers,
#        so their 14.3% bad rate is far below the truth.
# <2013: 153 loans before 2010, no 60-month product existed, crisis-era
#        lending rules. Different product, too few rows to learn from.
YEAR_MIN, YEAR_MAX = 2013, 2017

years = df['issue_d'].dt.year
df = df[(years >= YEAR_MIN) & (years <= YEAR_MAX)].copy().reset_index(drop=True)

print('rows after trim:', len(df))
print('bad rate       :', f"{df['target'].mean():.3%}")

# --- re-split on the trimmed data ----------------------------------------
cut = df['issue_d'].quantile(0.8)
print('cut-off date   :', cut.date())

train = df[df['issue_d'] <  cut].copy()
test  = df[df['issue_d'] >= cut].copy()

print(f"train rows: {len(train):6d}   share: {len(train)/len(df):.1%}   bad rate: {train['target'].mean():.3%}")
print(f"test  rows: {len(test):6d}   share: {len(test)/len(df):.1%}   bad rate: {test['target'].mean():.3%}")

assert len(train) + len(test) == len(df)
print('train:', train['issue_d'].min().date(), '->', train['issue_d'].max().date())
print('test :', test['issue_d'].min().date(), '->', test['issue_d'].max().date())

rows after trim: 26280
bad rate       : 20.784%
cut-off date   : 2016-09-01
train rows:  20902   share: 79.5%   bad rate: 20.046%
test  rows:   5378   share: 20.5%   bad rate: 23.652%
train: 2013-01-01 -> 2016-08-01
test : 2016-09-01 -> 2017-12-01


In [32]:
# assign it back, or nothing changes
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')

# exact whole months, measured from issue_d - never from today
df['credit_history_months'] = (
    (df['issue_d'].dt.year  - df['earliest_cr_line'].dt.year) * 12
    + (df['issue_d'].dt.month - df['earliest_cr_line'].dt.month)
)

print('negative values:', (df['credit_history_months'] < 0).sum())
print(df['credit_history_months'].describe())

negative values: 0
count    26280.000000
mean       197.671918
std         90.690052
min         37.000000
25%        136.000000
50%        180.000000
75%        243.000000
max        781.000000
Name: credit_history_months, dtype: float64


In [33]:
bands = pd.qcut(df['credit_history_months'], 10, duplicates='drop')
print(df.groupby(bands, observed=True)['target'].agg(['mean', 'size']))

                           mean  size
credit_history_months                
(36.999, 100.0]        0.234224  2694
(100.0, 127.0]         0.235227  2640
(127.0, 144.0]         0.225770  2631
(144.0, 161.0]         0.213405  2596
(161.0, 180.0]         0.208534  2695
(180.0, 201.0]         0.205894  2545
(201.0, 228.0]         0.190440  2636
(228.0, 263.0]         0.197844  2598
(263.0, 322.0]         0.183790  2628
(322.0, 781.0]         0.182270  2617


In [35]:
print(df['earliest_cr_line'].min())
print((df['credit_history_months'] > 600).sum())   # over 50 years
print(df.nlargest(5, 'credit_history_months')[['earliest_cr_line', 'issue_d', 'credit_history_months']])

1952-01-01 00:00:00
24
      earliest_cr_line    issue_d  credit_history_months
19205       1952-01-01 2017-02-01                    781
8136        1957-01-01 2017-09-01                    728
19566       1959-05-01 2017-01-01                    692
4672        1958-07-01 2015-06-01                    683
13046       1959-03-01 2016-02-01                    683


In [ ]:
CREDIT_HISTORY_CAP = 600   # 50 years - domain judgement, not read from the data
df['credit_history_months'] = df['credit_history_months'].clip(upper=CREDIT_HISTORY_CAP)

np.float64(0.0)

In [41]:
import numpy as np

df['loan_to_income']  = df['loan_amnt'] / df['annual_inc']
df['revol_to_income'] = df['revol_bal'] / df['annual_inc']

safe_total = df['total_acc'].replace(0, np.nan)
df['open_to_total_acc'] = df['open_acc'] / safe_total

new = ['loan_to_income', 'revol_to_income', 'open_to_total_acc']
print(df[new].describe().T)
print('\ninf values:', np.isinf(df[new]).sum().to_dict())
print('missing   :', df[new].isna().sum().to_dict())

                     count      mean       std       min       25%       50%  \
loan_to_income     26280.0  0.216257  0.169409  0.000436  0.125000  0.200000   
revol_to_income    26280.0  0.227474  0.224282  0.000000  0.100113  0.178729   
open_to_total_acc  26280.0  0.504475  0.175551  0.046154  0.375000  0.483871   

                        75%    max  
loan_to_income     0.291227  19.50  
revol_to_income    0.295309  10.76  
open_to_total_acc  0.615385   1.00  

inf values: {'loan_to_income': 0, 'revol_to_income': 0, 'open_to_total_acc': 0}
missing   : {'loan_to_income': 0, 'revol_to_income': 0, 'open_to_total_acc': 0}


In [42]:
for col in new:
    bands = pd.qcut(df[col], 10, duplicates='drop')
    print('\n---', col, '---')
    print(df.groupby(bands, observed=True)['target'].agg(['mean', 'size']))


--- loan_to_income ---
                         mean  size
loan_to_income                     
(-0.000564, 0.0755]  0.137590  2631
(0.0755, 0.111]      0.146332  2713
(0.111, 0.141]       0.176103  2561
(0.141, 0.169]       0.163218  2610
(0.169, 0.2]         0.185792  2928
(0.2, 0.232]         0.188817  2325
(0.232, 0.268]       0.227549  2628
(0.268, 0.316]       0.238204  2628
(0.316, 0.377]       0.287671  2628
(0.377, 19.5]        0.328387  2628

--- revol_to_income ---
                      mean  size
revol_to_income                 
(-0.001, 0.0507]  0.200533  2628
(0.0507, 0.085]   0.184551  2628
(0.085, 0.115]    0.200533  2628
(0.115, 0.146]    0.190639  2628
(0.146, 0.179]    0.203577  2628
(0.179, 0.216]    0.203196  2628
(0.216, 0.265]    0.219559  2628
(0.265, 0.333]    0.218417  2628
(0.333, 0.443]    0.221842  2628
(0.443, 10.76]    0.235540  2628

--- open_to_total_acc ---
                       mean  size
open_to_total_acc                
(0.0452, 0.292]    0.165781 

In [43]:
bands = pd.qcut(df['revol_util'], 10, duplicates='drop')
print(df.groupby(bands, observed=True)['target'].agg(['mean', 'size']))

                    mean  size
revol_util                    
(-0.001, 19.1]  0.168373  2637
(19.1, 29.7]    0.176021  2619
(29.7, 38.0]    0.187595  2628
(38.0, 45.4]    0.205177  2627
(45.4, 52.5]    0.204059  2661
(52.5, 59.4]    0.235384  2617
(59.4, 66.8]    0.215103  2622
(66.8, 74.8]    0.221286  2612
(74.8, 84.7]    0.230271  2623
(84.7, 162.0]   0.235137  2624


In [44]:
df = df.drop(columns='revol_to_income')

In [ ]:
# df['mths_since_last_delinq'].head(5)
# df['mths_since_last_delinq'].isna().sum() # np.int64(12900)
# df['mths_since_last_delinq'].value_counts().sum() #np.int64(13380)
# len(df['mths_since_last_delinq']) # 26280
# df['mths_since_last_delinq'].count() # np.int64(13380)
(df['mths_since_last_delinq'].nunique())

98